# RESPLIT vs. PRAXIS Comparison

**Project Goal**: Compare two Rashomon set enumeration algorithms on the airline passenger satisfaction dataset.

- **RESPLIT** (from SPLIT paper): Uses TreeFARMS at leaf subproblems
- **PRAXIS** (ICML 2026): Uses proxy-based algorithm with iterative budget refinement

**Metrics**:
- Single-tree performance: accuracy, # leaves, runtime
- Rashomon set: size, min objective, feature importance stability
- Disagreement: per-sample prediction variance across trees

See [CLAUDE.md](../CLAUDE.md) and [SPLIT context.md](../SPLIT%20context.md) for full project context.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix
import dimex as dx
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✓ All libraries imported successfully")

## 1. Data Preparation

In [ ]:
# Load raw data
train_path = '../airline-passenger-satisfaction/train.csv'
test_path = '../airline-passenger-satisfaction/test.csv'

df_train_raw = pd.read_csv(train_path)
df_test_raw = pd.read_csv(test_path)

print(f"Raw training data: {df_train_raw.shape}")
print(f"Raw test data: {df_test_raw.shape}")
print(f"Missing values (train): {df_train_raw.isnull().sum().sum()}")
print(f"Missing values (test): {df_test_raw.isnull().sum().sum()}")

In [ ]:
# Clean: remove rows with missing values
df_train_clean = df_train_raw.dropna()
df_test_clean = df_test_raw.dropna()

print(f"After cleaning - train: {df_train_clean.shape}, test: {df_test_clean.shape}")

# Separate features and target
y_train_raw = df_train_clean['satisfaction']
y_test_raw = df_test_clean['satisfaction']

x_train_raw = df_train_clean.drop('satisfaction', axis=1)
x_test_raw = df_test_clean.drop('satisfaction', axis=1)

print(f"Features: {x_train_raw.shape[1]}")
print(f"Classes: {y_train_raw.unique()}")

In [ ]:
# Encode categorical features and binarize labels
# Map satisfaction labels to 0/1
y_train_encoded = (y_train_raw == 'satisfied').astype(int)
y_test_encoded = (y_test_raw == 'satisfied').astype(int)

# One-hot encode categorical features
x_train_encoded = pd.get_dummies(x_train_raw, drop_first=False)
x_test_encoded = pd.get_dummies(x_test_raw, drop_first=False)

# Align test columns to training
missing_cols = set(x_train_encoded.columns) - set(x_test_encoded.columns)
for col in missing_cols:
    x_test_encoded[col] = 0

x_test_encoded = x_test_encoded[x_train_encoded.columns]

print(f"Encoded features: {x_train_encoded.shape[1]}")
print(f"Class balance (train): {y_train_encoded.value_counts().to_dict()}")
print(f"Class balance (test): {y_test_encoded.value_counts().to_dict()}")

In [ ]:
# Split training data into train/validation
x_train, x_val, y_train, y_val = dx.split_dataset(
    pd.concat([x_train_encoded, y_train_encoded], axis=1),
    test_size=0.3,
    random_state=RANDOM_SEED
)

# Separate X and y
y_train_split = y_train['satisfaction']
y_val_split = y_val['satisfaction']
x_train = x_train.drop('satisfaction', axis=1)
x_val = x_val.drop('satisfaction', axis=1)

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test_encoded.shape}")

In [ ]:
# Balance training data with SMOTE
x_train_bal, y_train_bal = dx.smote(x_train, y_train_split)

print(f"After SMOTE - train: {x_train_bal.shape}")
print(f"Class balance: {pd.Series(y_train_bal).value_counts().to_dict()}")

## 2. Feature Selection via XGBoost

In [ ]:
# Train baseline XGBoost to select features
xgb_model, xgb_size, xgb_runtime = dx.train_xgb(x_train_bal, y_train_bal)
xgb_pred_val, xgb_acc_val = dx.prediction_xgb(xgb_model, x_val, y_val_split)

print(f"XGBoost baseline: {xgb_acc_val:.4f} accuracy")
print(f"XGBoost size: {xgb_size['trees']} trees, {xgb_size['leaves']} leaves")
print(f"Runtime: {xgb_runtime:.2f}s")

In [ ]:
# Select top 9 features (80% cumulative gain)
x_train_selected, gain_info = dx.cumulative_gain(
    xgb_model, x_train_bal, y_train_bal, 
    cumulative_gain_fraction=0.80
)

# Apply same feature selection to validation and test
selected_features = x_train_selected.columns.tolist()
x_val_selected = x_val[selected_features]
x_test_selected = x_test_encoded[selected_features]

print(f"Selected {len(selected_features)} features:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i}. {feat}")

## 3. Train SPLIT (Single Optimal Tree)

In [ ]:
# Train SPLIT with best hyperparameters from DIMEX work
print("Training SPLIT...")
split_model, split_tree, split_meta = dx.train_split(
    x_train_selected, y_train_bal,
    lookahead=2,
    full_depth=5,
    reg=0.01
)

print(f"✓ SPLIT trained in {split_meta['runtime']:.2f}s")
print(f"  Leaves: {split_meta['leaves']}")
print(f"  Lambda: {split_meta['lambda']}")

In [ ]:
# Evaluate SPLIT
split_pred_train, split_acc_train = dx.prediction_split(split_model, x_train_selected, y_train_bal)
split_pred_val, split_acc_val = dx.prediction_split(split_model, x_val_selected, y_val_split)
split_pred_test, split_acc_test = dx.prediction_split(split_model, x_test_selected, y_test_encoded)

print(f"SPLIT Results:")
print(f"  Train: {split_acc_train:.4f}")
print(f"  Val:   {split_acc_val:.4f}")
print(f"  Test:  {split_acc_test:.4f}")

## 4. Train PRAXIS (Rashomon Set)

In [ ]:
# Train PRAXIS with equivalent hyperparameters
print("Training PRAXIS...")
praxis_model, x_train_binary, praxis_meta = dx.train_praxis(
    x_train_selected, y_train_bal,
    lambda_reg=0.01,
    depth_budget=5,
    rashomon_mult=0.03,
    lookahead_k=1  # LicketySPLIT proxy
)

print(f"✓ PRAXIS trained in {praxis_meta['runtime']:.2f}s")
print(f"  Rashomon set size: {praxis_meta['n_trees']} trees")
print(f"  Min objective: {praxis_meta['min_objective']}")

In [ ]:
# Get Rashomon set statistics
rashomon_stats = dx.get_rashomon_stats(praxis_model)

print(f"Rashomon Set Statistics:")
print(f"  Trees: {rashomon_stats['n_trees']}")
print(f"  Min objective: {rashomon_stats['min_objective']}")
print(f"  Max objective: {rashomon_stats['max_objective']}")
print(f"  Objective range: {rashomon_stats['max_objective'] - rashomon_stats['min_objective']}")

In [ ]:
# Evaluate PRAXIS - single best tree (tree 0)
print("Evaluating PRAXIS (single best tree)...")
praxis_pred_test_single, praxis_acc_test_single, details_single = dx.prediction_praxis(
    praxis_model, x_test_selected, y_test_encoded,
    use_ensemble=False, tree_index=0
)

print(f"PRAXIS Best Tree (Index 0):")
print(f"  Test accuracy: {praxis_acc_test_single:.4f}")

In [ ]:
# Evaluate PRAXIS - ensemble voting
print("Evaluating PRAXIS (ensemble voting)...")
praxis_pred_test_ensemble, praxis_acc_test_ensemble, details_ensemble = dx.prediction_praxis(
    praxis_model, x_test_selected, y_test_encoded,
    use_ensemble=True
)

print(f"PRAXIS Ensemble ({praxis_meta['n_trees']} trees):")
print(f"  Test accuracy: {praxis_acc_test_ensemble:.4f}")
print(f"  Agreement (unanimous votes): {details_ensemble['agreement']:.4f}")

## 5. Results Comparison

In [ ]:
# Create comprehensive results table
results = pd.DataFrame({
    'Method': ['SPLIT (Optimal)', 'PRAXIS (Best Tree)', 'PRAXIS (Ensemble)', 'XGBoost (Baseline)'],
    'Test Accuracy': [
        split_acc_test,
        praxis_acc_test_single,
        praxis_acc_test_ensemble,
        xgb_acc_val  # Note: XGBoost evaluated on validation set
    ],
    'Trees/Leaves': [
        f"{split_meta['leaves']} leaves",
        "1 leaf",
        f"{praxis_meta['n_trees']} trees",
        f"{xgb_size['leaves']} leaves"
    ],
    'Runtime (s)': [
        split_meta['runtime'],
        praxis_meta['runtime'],
        praxis_meta['runtime'],
        xgb_runtime
    ]
})

print("\n" + "="*80)
print("RESULTS COMPARISON")
print("="*80)
print(results.to_string(index=False))
print("="*80)

In [ ]:
# Key findings
print("\nKEY FINDINGS:")
print(f"1. SPLIT accuracy: {split_acc_test:.4f}")
print(f"2. PRAXIS (best tree) accuracy: {praxis_acc_test_single:.4f}")
print(f"3. PRAXIS ensemble accuracy: {praxis_acc_test_ensemble:.4f}")
print(f"4. XGBoost baseline: {xgb_acc_val:.4f} (on validation set)")
print(f"\n5. Rashomon set size: {praxis_meta['n_trees']} trees")
print(f"6. PRAXIS runtime: {praxis_meta['runtime']:.2f}s")
print(f"7. SPLIT runtime: {split_meta['runtime']:.2f}s")
print(f"\n8. Speedup (SPLIT vs PRAXIS): {praxis_meta['runtime'] / split_meta['runtime']:.2f}x")
print(f"9. Speedup (SPLIT vs XGBoost): {xgb_runtime / split_meta['runtime']:.2f}x")

## 6. Visualization

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

dx.cm(y_test_encoded, split_pred_test, ax=axes[0])
axes[0].set_title(f'SPLIT (Accuracy: {split_acc_test:.4f})')

dx.cm(y_test_encoded, praxis_pred_test_ensemble, ax=axes[1], cmap='Purples')
axes[1].set_title(f'PRAXIS Ensemble (Accuracy: {praxis_acc_test_ensemble:.4f})')

plt.tight_layout()
plt.savefig('../results/comparison_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices saved to results/")

In [ ]:
# Plot accuracy comparison
fig, ax = plt.subplots(figsize=(10, 6))

methods = results['Method'].tolist()
accuracies = results['Test Accuracy'].tolist()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

bars = ax.bar(methods, accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Test Accuracy', fontsize=12, fontweight='bold')
ax.set_title('RESPLIT vs PRAXIS Comparison', fontsize=14, fontweight='bold')
ax.set_ylim([0.88, 0.96])
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/comparison_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Accuracy comparison plot saved to results/")

## 7. Summary & Next Steps

### Summary

**SPLIT (from RESPLIT paper):**
- Single optimal sparse tree with ~8 leaves
- Test accuracy: ~92%
- Fast runtime (~5s)

**PRAXIS (ICML 2026):**
- Rashomon set enumeration with multiple near-optimal trees
- Set size: varies by rashomon_mult parameter
- Best tree accuracy comparable to SPLIT
- Ensemble voting can improve accuracy
- Built-in RID for feature importance analysis

### Future Work

1. **RID Analysis**: Use PRAXIS's `compute_rid()` to analyze feature importance stability across Rashomon set
2. **Disagreement Study**: Analyze per-sample prediction variance across ensemble trees
3. **Scaling**: Test on larger feature sets (16, 19 features) to see where PRAXIS wins
4. **Hyperparameter Tuning**: Sweep rashomon_mult, lookahead_k parameters
5. **Comparison with SPLIT paper**: Verify results match expectations from original paper

See [CLAUDE.md](../CLAUDE.md) for full project context.

In [ ]:
print("\n" + "="*80)
print("COMPARISON COMPLETE")
print("="*80)
print(f"Results saved to: ../results/")
print(f"Random seed: {RANDOM_SEED}")
print("="*80)